# Import Library


In [ ]:
import os
import numpy as np
import pandas as pd
import cv2
import random
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow as tf
import gc
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score, recall_score, f1_score
from tensorflow.keras.applications import VGG16
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.applications.vgg16 import preprocess_input
import xgboost as xgb
from tqdm import tqdm
import time
import hashlib

!pip install optuna --quiet
import optuna
from optuna.samplers import TPESampler, RandomSampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Load Dataset

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input/datasets"):
    print(root)

In [ ]:
DATA_DIR = "/kaggle/input/datasets/nazwasophiane/dataset-cor/data coronal"

paths = []
labels = []

for cls in os.listdir(DATA_DIR):
    cls_path = os.path.join(DATA_DIR, cls)
    if os.path.isdir(cls_path):
        for img_name in os.listdir(cls_path):
            paths.append(os.path.join(cls_path, img_name))
            labels.append(cls)

df = pd.DataFrame({"path": paths, "label": labels})

In [ ]:
# Cek file duplikat berdasarkan nama file
print("Mengecek file duplikat berdasarkan nama file...")

file_names = [os.path.basename(path) for path in paths]
df_files   = pd.DataFrame({'path': paths, 'filename': file_names, 'label': labels})

# Cari nama file yang muncul lebih dari sekali
duplicates = df_files[df_files.duplicated(subset='filename', keep=False)]

print(f"Total gambar   : {len(paths)}")
print(f"Total duplikat : {len(duplicates)}")

if len(duplicates) > 0:
    print("\nDaftar file duplikat:")
    print(duplicates.sort_values('filename').to_string(index=False))
else:
    print("Tidak ada file duplikat ditemukan.")

In [ ]:
# Cek data redundan

def file_hash(path, chunk_size=8192):
    hash_md5 = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            hash_md5.update(chunk)
    return hash_md5.hexdigest()

# Build hash dictionary sekali
print("Menghitung hash untuk semua file...")
hash_dict = {}
for path in df["path"]:
    h = file_hash(path)
    hash_dict.setdefault(h, []).append(path)

# Deteksi duplikat
duplicate_groups = {h: paths for h, paths in hash_dict.items() if len(paths) > 1}
print(f"Jumlah grup duplikat konten: {len(duplicate_groups)}")

# Simpan daftar duplikat ke CSV format PIVOT (1 baris = 1 grup, tiap file jadi kolom)
rows = []
for h, paths in duplicate_groups.items():
    filenames = [os.path.basename(p) for p in paths]
    row = {}
    for i, fname in enumerate(filenames, start=1):
        row[f"file_{i}"] = fname
    rows.append(row)

df_duplikat = pd.DataFrame(rows)
df_duplikat.to_csv("/kaggle/working/daftar_duplikat_coronall.csv", index=False)
print(f"Total grup duplikat disimpan ke CSV: {len(df_duplikat)}")

# Filter df - ambil 1 path pertama dari tiap hash
df_before = len(df)
unique_paths = [paths[0] for paths in hash_dict.values()]
df = df[df["path"].isin(unique_paths)].reset_index(drop=True)

print(f"\nJumlah gambar sebelum dedup : {df_before}")
print(f"Jumlah gambar setelah dedup : {len(df)}")
print(f"Jumlah yang dibuang         : {df_before - len(df)}")

# EDA

In [ ]:
# EDA 1 & 2: Validasi jumlah data per kelas dan label
print("Jumlah data per kelas:")
for label in df['label'].unique():
    total = df[df['label'] == label].shape[0]
    print(f"Kelas {label}: {total} citra")

# EDA 3: Visualisasi distribusi kelas 
plt.figure(figsize=(10, 6))
ax = sns.countplot(x='label', data=df, palette='viridis')

for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center',
                xytext=(0, 9),
                textcoords='offset points')

plt.title('Distribusi Jumlah Data Per Kelas')
plt.xlabel('Nama Kelas')
plt.ylabel('Jumlah Citra')
plt.show()
plt.close()

In [ ]:
# Visualisasi sampel citra per kelas
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, label in enumerate(['normal seluruh tubuh', 'stone seluruh tubuh', 'tumor seluruh tubuh', 'cyst seluruh tubuh']):
    sample_img = df[df['label'] == label].iloc[0]['path']
    img = cv2.imread(sample_img, cv2.IMREAD_GRAYSCALE)
    axes[i].imshow(img, cmap='gray')
    axes[i].set_title(f'{label}')
    axes[i].axis('off')
plt.suptitle('Sampel Citra CT Scan per Kelas')
plt.show()

# Persiapan Data

In [ ]:
# Encoding Label

le = LabelEncoder()
y  = le.fit_transform(df["label"])

print("Mapping kelas:", dict(zip(le.classes_, le.transform(le.classes_))))
print("Contoh sebelum :", df["label"].head().tolist())
print("Contoh sesudah :", y[:5])
df["label"].value_counts()

In [ ]:
# Cek jumlah channel tiap gambar
# VGG16 butuh input 3 channel (RGB), jadi perlu dipastikan semua gambar sudah berformat 3 channel sebelum diproses

channel_count = {}

for path in df["path"]:
    img = cv2.imread(path)
    if img is None:
        continue
    shape = img.shape
    ch = 1 if len(shape) == 2 else shape[2]
    if ch not in channel_count:
        channel_count[ch] = 0
    channel_count[ch] += 1

print("Jumlah citra berdasarkan jumlah channel:")
for ch, total in channel_count.items():
    print(f"{ch} channel: {total} citra")

print("\nShape unik yang ditemukan:")
unique_shapes = set([cv2.imread(p).shape for p in df["path"] if cv2.imread(p) is not None])
print(unique_shapes)

# Preprocessing — Semua Data (Sebelum Split)

> Preprocessing dilakukan ke **seluruh dataset** sebelum split.
> Output: `X_all_images` array gambar ternormalisasi shape (N, 224, 224, 3).
> Preprocessing menggunakan generator batch per batch agar hemat RAM saat proses,
> namun hasil akhirnya tetap disimpan sebagai array lengkap untuk di-split.

In [ ]:
# Visualisasi preprocessing pada 1 sampel (sebelum generator dijalankan)
sample_path = df["path"].iloc[0]
sample_raw  = cv2.imread(sample_path)
sample_raw  = cv2.resize(sample_raw, (224, 224))
sample_rgb  = cv2.cvtColor(sample_raw, cv2.COLOR_BGR2RGB)
sample_norm = preprocess_input(sample_rgb.astype(np.float32))

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(sample_rgb)
axes[0].set_title(f"Sebelum Preprocessing\nMin:{sample_rgb.min()}  Max:{sample_rgb.max()}")
axes[0].axis("off")

disp = (sample_norm - sample_norm.min()) / (sample_norm.max() - sample_norm.min())
axes[1].imshow(disp)
axes[1].set_title(f"Sesudah Preprocessing (Normalisasi VGG16)\nMin:{sample_norm.min():.2f}  Max:{sample_norm.max():.2f}")
axes[1].axis("off")

plt.suptitle("Visualisasi Tahap Preprocessing Citra", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
def preprocess_all_images(path_list, label_list, batch_size=32):
    all_images = []
    all_labels = []
    all_paths  = []  
    total      = len(path_list)
    for start in range(0, total, batch_size):
        batch_paths  = path_list[start:start+batch_size]
        batch_labels = label_list[start:start+batch_size]
        for path, lbl in zip(batch_paths, batch_labels):
            img = cv2.imread(path)
            if img is None:
                print(f"  [SKIP] Gagal baca: {path}")
                continue
            img  = cv2.resize(img, (224, 224))
            rgb  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            norm = preprocess_input(rgb.astype(np.float32))
            all_images.append(norm)
            all_labels.append(lbl)
            all_paths.append(path) 
        if (start // batch_size) % 10 == 0:
            print(f"  Diproses: {min(start+batch_size, total)}/{total} gambar", end="\r")
    print(f"  Selesai  : {total}/{total} gambar")
    return np.array(all_images), np.array(all_labels), np.array(all_paths) 

print("Preprocessing seluruh dataset...")
print(f"Total gambar   : {len(df)}")

all_paths_input  = np.array(df["path"].tolist())
all_labels_input = y

X_all_images, y_all, all_paths_valid = preprocess_all_images(all_paths_input, all_labels_input)

print(f"\nShape X_all_images : {X_all_images.shape}")
if len(all_paths_valid) != len(all_paths_input):
    print(f"{len(all_paths_input) - len(all_paths_valid)} gambar gagal dibaca dan di-skip.")

# Split Data (Setelah Preprocessing)

> Split dilakukan pada `X_all_images` yang sudah dipreprocess — bukan path mentah.
> Path di-split paralel untuk keperluan augmentasi train.

In [ ]:
# Split array gambar ternormalisasi — Train+Val 85% | Test 15%
paths_arr = all_paths_valid

# Split array gambar (untuk val dan test)
X_trainval_images, X_test_images, y_trainval, y_test = train_test_split(
    X_all_images, y_all,
    test_size=0.15,
    random_state=42,
    stratify=y_all
)

# Split path paralel (untuk augmentasi train)
train_val_paths, test_paths, train_val_y, test_y = train_test_split(
    paths_arr, y_all,
    test_size=0.15,
    random_state=42,
    stratify=y_all
)

# Bebaskan X_all_images dari RAM setelah split
del X_all_images
gc.collect()

print(f"Train+Val images : {X_trainval_images.shape}")
print(f"Test images      : {X_test_images.shape}")
print(f"\nDistribusi kelas di test set:")
for cls_idx, cls_name in enumerate(le.classes_):
    print(f"  {cls_name}: {np.sum(y_test == cls_idx)} citra")

In [ ]:
# K-Fold
kf         = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_splits = list(kf.split(train_val_paths, train_val_y))

print(f"{'Fold':<6} {'Train':>8} {'Val':>8} {'Test':>8}")
print("-" * 35)
for fold, (train_idx, val_idx) in enumerate(fold_splits):
    print(f"Fold {fold+1}  {len(train_idx):>8} {len(val_idx):>8} {len(test_paths):>8}")
print("-" * 35)
print(f"Test set: {len(test_paths)} ({len(test_paths)/(len(train_val_paths)+len(test_paths))*100:.0f}%)")

In [ ]:
# Cek distribusi per kelas di setiap fold, emastikan setiap kelas terwakili secara proporsional

class_names_split = sorted([cls for cls in os.listdir(DATA_DIR)
                            if os.path.isdir(os.path.join(DATA_DIR, cls))])

print(f"{'':8} {'':12} {'Total':>7}", end="")
for cls in class_names_split:
    print(f" {cls:>8}", end="")
print()

for fold, (train_idx, val_idx) in enumerate(fold_splits):
    for split_name, idx in [('Train', train_idx), ('Val', val_idx)]:
        y_split = train_val_y[idx]
        print(f"Fold {fold+1}   {split_name:12} {len(idx):>7}", end="")
        for cls_idx in range(4):
            print(f" {np.sum(y_split == cls_idx):>8}", end="")
        print()

print(f"Test     {'':12} {len(test_paths):>7}", end="")
for cls_idx in range(4):
    print(f" {np.sum(test_y == cls_idx):>8}", end="")
print()

# Load VGG16

In [ ]:
# Load VGG16 sebagai feature extractor
# trainable=False → semua layer VGG16 dikunci, tidak dilatih ulang
# GlobalAveragePooling2D → ubah output conv (7×7×512) menjadi vektor 512 dimensi

base_model = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)
base_model.trainable = False  # Freeze semua layer VGG16

x_gap = GlobalAveragePooling2D(name='gap')(base_model.output)

feature_extractor = Model(
    inputs=base_model.input,
    outputs=x_gap,
    name='VGG16_GAP_Extractor'
)
feature_extractor.summary()

# Augmentasi Data Train

In [ ]:
# Fungsi sharpening: mempertajam detail/tepi gambar
# Cara kerja: gambar asli dikurangi versi blur-nya sehingga detail yang halus menjadi lebih terlihat

def apply_sharpening(img_bgr):
    blur  = cv2.GaussianBlur(img_bgr, (0, 0), sigmaX=2)
    sharp = cv2.addWeighted(img_bgr, 1.5, blur, -0.5, 0)
    return sharp

In [ ]:
# Fungsi augmentasi — HANYA digunakan untuk data train
# 3 teknik augmentasi:
# Rotation  : putar gambar acak antara -10° hingga +10°
# Brightness: ubah kecerahan dengan faktor acak 0.7-1.3
# Sharpening: pertajam detail gambar (nilai tetap, tidak acak)

def augment_train(img_bgr):
    h, w = img_bgr.shape[:2]

    # Rotation ±15 derajat
    angle   = random.uniform(-15, 15)
    M       = cv2.getRotationMatrix2D((w//2, h//2), angle, 1.0)
    img_bgr = cv2.warpAffine(img_bgr, M, (w, h), borderMode=cv2.BORDER_REFLECT)

    # Brightness ×0.1 sampai ×1.5
    factor    = random.uniform(0.1, 1.5)
    img_float = img_bgr.astype(np.float32) * factor
    img_bgr   = np.clip(img_float, 0, 255).astype(np.uint8)

    # Sharpening (nilai tetap)
    img_bgr = apply_sharpening(img_bgr)

    return img_bgr # dikembalikan 1 gambar hasil ketiga teknik

In [ ]:
# Visualisasi hasil augmentasi, tampilkan gambar asli vs 3 teknik augmentasi

sample_path = df['path'].iloc[0]
raw_orig    = cv2.imread(sample_path)
raw_orig    = cv2.resize(raw_orig, (224, 224))
rgb_orig    = cv2.cvtColor(raw_orig, cv2.COLOR_BGR2RGB)

# Rotation saja
h, w    = raw_orig.shape[:2]
angle   = 10
M       = cv2.getRotationMatrix2D((w//2, h//2), angle, 1.0)
raw_rot = cv2.warpAffine(raw_orig, M, (w, h), borderMode=cv2.BORDER_REFLECT)
rgb_rot = cv2.cvtColor(raw_rot, cv2.COLOR_BGR2RGB)

# Brightness saja
factor     = 1.3
raw_bright = np.clip(raw_orig.astype(np.float32) * factor, 0, 255).astype(np.uint8)
rgb_bright = cv2.cvtColor(raw_bright, cv2.COLOR_BGR2RGB)

# Sharpening saja
raw_sharp = apply_sharpening(raw_orig.copy())
rgb_sharp = cv2.cvtColor(raw_sharp, cv2.COLOR_BGR2RGB)

images = [rgb_orig, rgb_rot, rgb_bright, rgb_sharp]
titles = ['Asli (Original)', 'Rotation (+15°)', 'Brightness (×1.5)', 'Sharpening']
colors = ['gray', '#1f77b4', '#ff7f0e', '#2ca02c']

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for idx, (img, title, color) in enumerate(zip(images, titles, colors)):
    axes[idx].imshow(img)
    axes[idx].set_title(title, fontsize=12, fontweight='bold', color=color, pad=10)
    axes[idx].axis('off')

plt.suptitle('Visualisasi Augmentasi Data Train', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Keterangan augmentasi:")
print("  1. Rotation   : gambar dirotasi secara acak antara -15° hingga +15°")
print("  2. Brightness : kecerahan dikalikan faktor acak antara 0.1 hingga 1.5")
print("  3. Sharpening : detail/tepi gambar dipertajam menggunakan unsharp masking")


In [ ]:
# Fungsi augmentasi adaptif per kelas (hanya untuk data TRAIN)
# Input : path gambar train fold + labelnya
# Output: path gambar asli + augmentasi

def augment_minority_paths(path_list, label_list, fold_id=""):
    path_list  = np.array(path_list)
    label_list = np.array(label_list)

    classes, counts = np.unique(label_list, return_counts=True)
    target = int(counts.max())

    print(f"\n{'='*50}")
    print(f"Distribusi kelas sebelum augmentasi {fold_id}:")
    ada_augmentasi = False
    for cls, cnt in zip(classes, counts):
        nama    = le.classes_[cls]
        deficit = target - cnt
        status  = "mayoritas tidak diaugmentasi" if deficit == 0 else f"minoritas — deficit {deficit} citra"
        print(f"  {nama:20}: {cnt:5} citra | {status}")
        if deficit > 0:
            ada_augmentasi = True
    print(f"  Target per kelas : {target}")

    if not ada_augmentasi:
        print("  → Semua kelas sudah seimbang, tidak ada augmentasi")
        return path_list, label_list

    aug_paths  = list(path_list)
    aug_labels = list(label_list)

    for cls, cnt in zip(classes, counts):
        deficit = target - cnt
        if deficit <= 0:
            continue

        cls_paths = path_list[label_list == cls]
        generated = 0

        while generated < deficit:
            src_path = cls_paths[np.random.randint(len(cls_paths))]
            img = cv2.imread(src_path)
            if img is None:
                continue
            img = cv2.resize(img, (224, 224))
            aug = augment_train(img.copy())

            save_dir  = f"/kaggle/working/aug_cls_{cls}_{fold_id}"
            os.makedirs(save_dir, exist_ok=True)
            save_path = os.path.join(save_dir, f"aug_{generated}.jpg")
            cv2.imwrite(save_path, aug)

            aug_paths.append(save_path)
            aug_labels.append(cls)
            generated += 1

    aug_paths  = np.array(aug_paths)
    aug_labels = np.array(aug_labels)

    print(f"\nDistribusi kelas setelah augmentasi {fold_id}:")
    for cls in classes:
        nama  = le.classes_[cls]
        total = int(np.sum(aug_labels == cls))
        print(f"  {nama:20}: {total:5} citra")
    print(f"{'='*50}")

    return aug_paths, aug_labels

In [ ]:
# Fungsi ekstraksi fitur VGG16 dari path
# Dipakai HANYA untuk data train yang sudah diaugmentasi
# Val dan test diekstrak dari array gambar (X_trainval_images, X_test_images)

def extract_features_from_paths(path_list, label_list, batch_size=32):
    all_features = []
    all_labels   = []

    for i in range(0, len(path_list), batch_size):
        batch_paths  = path_list[i:i+batch_size]
        batch_labels = label_list[i:i+batch_size]

        batch_imgs = []
        for path in batch_paths:
            img = cv2.imread(path)
            if img is None:
                continue
            img  = cv2.resize(img, (224, 224))
            rgb  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            norm = preprocess_input(rgb.astype(np.float32))
            batch_imgs.append(norm)

        if batch_imgs:
            feats = feature_extractor.predict(np.array(batch_imgs), verbose=0)
            all_features.append(feats)
            all_labels.extend(batch_labels[:len(batch_imgs)])

    return np.concatenate(all_features), np.array(all_labels)

In [ ]:
# Ekstraksi fitur val dan test dari array gambar yang sudah dipreprocess
# Dilakukan sekali di sini — tidak perlu diulang per fold
print("Ekstraksi fitur val (dari array gambar ternormalisasi)...")
X_trainval = feature_extractor.predict(X_trainval_images, batch_size=32, verbose=1)
print(f"  Shape X_trainval : {X_trainval.shape}")

print("\nEkstraksi fitur test (dari array gambar ternormalisasi)...")
X_test = feature_extractor.predict(X_test_images, batch_size=32, verbose=1)
print(f"  Shape X_test     : {X_test.shape}")

# Bebaskan array gambar dari RAM setelah fitur diekstrak
del X_trainval_images, X_test_images
gc.collect()

In [ ]:
# Visualisasi & Simpan Hasil Ekstraksi Fitur VGG16
class_names_all = sorted([
    cls for cls in os.listdir(DATA_DIR)
    if os.path.isdir(os.path.join(DATA_DIR, cls))
])

df_features_full = pd.DataFrame(
    X_test,
    columns=[f'Fitur_{i+1}' for i in range(X_test.shape[1])]
)
df_features_full.insert(0, 'label', [class_names_all[lbl] for lbl in y_test])
df_features_full.insert(0, 'gambar', [f'gambar_{i+1}' for i in range(len(X_test))])

print("Contoh 10 baris pertama, 10 fitur pertama:")
print(df_features_full.iloc[:10, :12].to_string(index=False))
print(f"\nNilai min  : {X_test.min():.4f}")
print(f"Nilai max  : {X_test.max():.4f}")
print(f"Nilai mean : {X_test.mean():.4f}")

plt.figure(figsize=(14, 5))
sns.heatmap(
    X_test[:10, :20],
    cmap='Blues',
    annot=True, fmt='.2f',
    xticklabels=[f'F{i+1}' for i in range(20)],
    yticklabels=[f'Gambar {i+1}' for i in range(10)],
    linewidths=0.5
)
plt.title('Heatmap Hasil Ekstraksi Fitur VGG16 (Test Set)\n(10 Sampel x 20 Fitur Pertama)', fontweight='bold')
plt.xlabel('Fitur VGG16')
plt.ylabel('Sampel Gambar')
plt.tight_layout()
plt.show()

csv_path = '/kaggle/working/hasil_ekstraksi_fitur_vgg16.csv'
df_features_full.to_csv(csv_path, index=False)
print(f"\nDisimpan ke {csv_path}")
print(f"   Total baris  : {len(df_features_full)} gambar")
print(f"   Total kolom  : {df_features_full.shape[1]} (gambar + label + 512 fitur)")

# Training XGBoost

In [ ]:
# TRAINING XGBOOST CROSS VALIDATION (K-Fold)
# Augmentasi adaptif — hanya kelas minoritas di tiap train fold
# Val  → X_trainval[val_idx] dari generator (sudah dipreprocess sebelum split)
# Test → X_test dari generator (sudah dipreprocess sebelum split)

fold_results = []
start_cv     = time.time()

for fold, (train_idx, val_idx) in enumerate(fold_splits):
    train_paths_fold  = train_val_paths[train_idx]
    train_labels_fold = train_val_y[train_idx]

    print(f"\n{'#'*50}")
    print(f"FOLD {fold+1}")
    start_fold = time.time()

    # Augmentasi kelas minoritas — hanya train fold
    aug_paths, aug_labels = augment_minority_paths(
        train_paths_fold, train_labels_fold, fold_id=f"fold{fold+1}"
    )

    # Ekstraksi fitur train (asli + augmentasi)
    X_train, y_train = extract_features_from_paths(aug_paths, aug_labels)
    print(f"  Shape X_train (setelah aug): {X_train.shape}")

    # Val — dari X_trainval hasil generator, tidak diaugmentasi
    X_val = X_trainval[val_idx]
    y_val = y_trainval[val_idx]
    print(f"  Shape X_val               : {X_val.shape}")

    model_fold = xgb.XGBClassifier(
       n_estimators=581,
        max_depth=4,
        learning_rate=0.089591,
        subsample=0.890828,
        colsample_bytree=0.840371,
        gamma=0.013091,
        reg_alpha=0.000005,
        reg_lambda=0.25841,
        min_child_weight=8,
        objective="multi:softprob", 
        num_class=4,
        eval_metric=["mlogloss", "merror"], 
        tree_method="hist",
        early_stopping_rounds=20, 
        random_state=42, 
        verbosity=0, 
        n_jobs=-1
    )
    model_fold.fit(
        X_train, y_train,
        eval_set=[(X_train, y_train), (X_val, y_val)],
        verbose=False
    )

    fold_train_acc = accuracy_score(y_train, model_fold.predict(X_train))
    fold_val_acc   = accuracy_score(y_val,   model_fold.predict(X_val))

    del X_train, y_train
    gc.collect()

    elapsed_fold = time.time() - start_fold
    fold_results.append({
        "fold"          : fold+1,
        "train_accuracy": round(fold_train_acc, 6),
        "val_accuracy"  : round(fold_val_acc, 6),
        "waktu_detik"   : round(elapsed_fold, 2),
    })
    print(f"Fold {fold+1} selesai — Train: {fold_train_acc:.4f} | Val: {fold_val_acc:.4f} | Waktu: {elapsed_fold:.1f}s")

elapsed_cv = time.time() - start_cv
df_fold    = pd.DataFrame(fold_results).set_index("fold")
print("\n", df_fold.to_string())
print(f"\nRata-rata Train Acc : {df_fold['train_accuracy'].mean():.6f}")
print(f"Rata-rata Val Acc   : {df_fold['val_accuracy'].mean():.6f}")
print(f"Std Dev Val Acc     : {df_fold['val_accuracy'].std():.6f}")
print(f"Total waktu K-Fold  : {elapsed_cv//60:.0f} menit {elapsed_cv%60:.1f} detik")

In [ ]:
# TRAINING FINAL
start_final = time.time()

# Pisahkan train final dan early stopping dari train_val
train_final_paths, early_stop_paths, train_final_labels, early_stop_labels = train_test_split(
    train_val_paths, train_val_y,
    test_size=0.15, stratify=train_val_y, random_state=42
)

# Early stopping — ambil dari X_trainval menggunakan mapping path → indeks
path_to_idx  = {path: i for i, path in enumerate(train_val_paths)}
early_stop_indices = [path_to_idx[path] for path in early_stop_paths]
X_early_stop = X_trainval[early_stop_indices]
y_early_stop = early_stop_labels

print(f"  Shape train_final : {len(train_final_paths)} citra")
print(f"  Shape X_early_stop: {X_early_stop.shape}")

# Augmentasi kelas minoritas pada train final
print("\nAugmentasi kelas minoritas pada train final...")
aug_final_paths, aug_final_labels = augment_minority_paths(
    train_final_paths, train_final_labels, fold_id="final"
)

# Ekstraksi fitur train final (asli + augmentasi)
print("\nEkstraksi fitur train final...")
X_train_final, y_train_final = extract_features_from_paths(aug_final_paths, aug_final_labels)
print(f"  Shape X_train_final : {X_train_final.shape}")

# Untuk evaluasi train
X_trainval_final = X_train_final
y_trainval_final = y_train_final

# Training model — hyperparameter SAMA dengan K-Fold
start_train = time.time()
model_xgb = xgb.XGBClassifier(
    n_estimators=581,
        max_depth=4,
        learning_rate=0.089591,
        subsample=0.890828,
        colsample_bytree=0.840371,
        gamma=0.013091,
        reg_alpha=0.000005,
        reg_lambda=0.25841,
        min_child_weight=8,
    objective="multi:softprob",
    num_class=4,
    eval_metric=["mlogloss", "merror"],
    tree_method="hist",
    early_stopping_rounds=20,
    random_state=42,
    verbosity=0,
    n_jobs=-1
)
model_xgb.fit(
    X_train_final, y_train_final,
    eval_set=[(X_train_final, y_train_final), (X_early_stop, y_early_stop)],
    verbose=False
)

elapsed_train = time.time() - start_train
elapsed_final = time.time() - start_final

print("\nTraining final selesai.")
print(f"Train Accuracy : {accuracy_score(y_train_final, model_xgb.predict(X_train_final)):.4f}")
print(f"Test  Accuracy : {accuracy_score(y_test, model_xgb.predict(X_test)):.4f}")
print(f"Waktu XGBoost  : {elapsed_train//60:.0f} menit {elapsed_train%60:.1f} detik")
print(f"Pohon optimal  : {model_xgb.best_iteration + 1}")

In [ ]:
results = model_xgb.evals_result()

In [ ]:
# Learning Curve — tampilkan grafik loss dan accuracy per boosting round
# Digunakan untuk mendeteksi overfitting

epochs = len(results['validation_0']['mlogloss'])
x_axis = range(epochs)

train_acc = [1 - e for e in results['validation_0']['merror']]
val_acc   = [1 - e for e in results['validation_1']['merror']]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Grafik kiri — Loss
axes[0].plot(x_axis, results['validation_0']['mlogloss'],
             label='Train Loss', color='#1f77b4')
axes[0].plot(x_axis, results['validation_1']['mlogloss'],
             label='Val Loss', color='#ff7f0e')
axes[0].set_title('Train vs Val Loss', fontweight='bold')
axes[0].set_xlabel('Boosting Round')
axes[0].set_ylabel('Log Loss')
axes[0].legend()

# Grafik kanan — Accuracy
axes[1].plot(x_axis, train_acc,
             label='Train Accuracy', color='#1f77b4')
axes[1].plot(x_axis, val_acc,
             label='Val Accuracy', color='#ff7f0e')
axes[1].set_title('Train vs Val Accuracy', fontweight='bold')
axes[1].set_xlabel('Boosting Round')
axes[1].set_ylabel('Accuracy')
axes[1].legend()

plt.suptitle('Learning Curve Hybrid VGG16-XGBoost',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Evaluasi model pada data train dan data test
# Train: untuk cek apakah model belajar dengan baik
# Test : untuk cek performa model pada data yang belum pernah dilihat

# TRAIN 
y_trainval_pred = model_xgb.predict(X_trainval_final)
print(" TRAIN ")
print("Accuracy:", accuracy_score(y_trainval_final, y_trainval_pred))
print(confusion_matrix(y_trainval_final, y_trainval_pred))
print(classification_report(y_trainval_final, y_trainval_pred))

# VAL 
print("\n VALIDASI (Cross Validation) ")
print(f"Rata-rata Val Accuracy : {df_fold['val_accuracy'].mean():.6f}")
print(f"Std Dev Val Accuracy   : {df_fold['val_accuracy'].std():.6f}")
print(df_fold.to_string())

# TEST 
y_test_pred = model_xgb.predict(X_test)
print("\n TEST ")
print("Accuracy:", accuracy_score(y_test, y_test_pred))
print(confusion_matrix(y_test, y_test_pred))
print(classification_report(y_test, y_test_pred))

In [ ]:
# Confusion Matrix — visualisasi sebaran prediksi benar dan salah
# Diagonal = prediksi benar | Di luar diagonal = prediksi salah

class_names = sorted([
    cls for cls in os.listdir(DATA_DIR)
    if os.path.isdir(os.path.join(DATA_DIR, cls))
])

def plot_confusion_matrix(y_true, y_pred, classes, title='Confusion Matrix'):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(9, 7))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=classes, yticklabels=classes)
    plt.xlabel('Prediksi Model')
    plt.ylabel('Label Sebenarnya')
    plt.title(title)
    plt.show()

# Confusion Matrix TRAIN
plot_confusion_matrix(y_trainval_final, y_trainval_pred, class_names,
                      title='Confusion Matrix Train')

# Confusion Matrix TEST
plot_confusion_matrix(y_test, y_test_pred, class_names,
                      title='Confusion Matrix Test')


In [ ]:
# Heatmap Classification Report
# Precision, Recall, F1-Score per kelas

report = classification_report(
    y_test, y_test_pred,
    target_names=class_names,
    output_dict=True
)

df_report = pd.DataFrame(report).T
df_report = df_report.loc[class_names, ['precision', 'recall', 'f1-score']]

plt.figure(figsize=(8, 5))
sns.heatmap(
    df_report.astype(float),
    annot=True, fmt='.4f',
    cmap='Blues',
    vmin=0, vmax=1,
    linewidths=0.5
)
plt.title('Classification Report Hybrid VGG16-XGBoost (Test Set)', fontweight='bold')
plt.ylabel('Kelas')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Identifikasi gambar yang salah diklasifikasi
# Disimpan ke CSV untuk analisis lebih lanjut

test_paths_list = list(test_paths)  # konversi numpy array ke list

misclassified = []
for i in range(len(y_test)):
    true_lbl = class_names[y_test[i]]
    pred_lbl = class_names[int(y_test_pred[i])]
    if true_lbl != pred_lbl:
        misclassified.append({
            "file_path" : test_paths_list[i],
            "file_name" : os.path.basename(test_paths_list[i]),
            "true_label": true_lbl,
            "pred_label": pred_lbl
        })

df_miss = pd.DataFrame(misclassified)
print(f"Total misklasifikasi : {len(df_miss)} dari {len(y_test)} data test")

if len(df_miss) == 0:
    print("Tidak ada misklasifikasi model mengklasifikasikan semua data test dengan benar.")
else:
    print(f"\nJumlah per kombinasi kelas:")
    print(df_miss.groupby(["true_label", "pred_label"]).size().reset_index(name="jumlah").to_string(index=False))
    df_miss.to_csv("/kaggle/working/misclassified_files.csv", index=False)
    print(f"\nDisimpan ke /kaggle/working/misclassified_files.csv")
    print(df_miss.to_string(index=False))

In [ ]:
# Visualisasi 10 sampel prediksi dari data test
# Hijau = prediksi benar | Merah = prediksi salah

plt.figure(figsize=(14, 6))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    img = cv2.imread(test_paths_list[i])
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.imshow(img)
    plt.axis("off")
    true_lbl = class_names[y_test[i]]
    pred_lbl = class_names[int(y_test_pred[i])]
    color    = "green" if true_lbl == pred_lbl else "red"
    plt.title(f"Asli : {true_lbl}\nPred : {pred_lbl}", fontsize=7, color=color)

plt.suptitle("Prediksi Model Hybrid VGG16-XGBoost (10 Sampel Test)", fontweight="bold")
plt.tight_layout()
plt.show()